# Orpheus Turkish TTS Fine-Tuning with LoRA

This notebook fine-tunes [Orpheus-3B](https://huggingface.co/unsloth/orpheus-3b-0.1-pretrained) for **Turkish text-to-speech** using LoRA and the `TransformersTrainer` SDK on **Red Hat OpenShift AI**.

## Overview

Orpheus-3B is a codec language model that generates speech as discrete SNAC audio tokens. This example:

1. **Preprocesses** Turkish audio into SNAC token sequences (rank 0, inside the TrainJob)
2. **Fine-tunes** using LoRA + DDP across multiple nodes via `TransformersTrainer`
3. **Merges** the LoRA adapter into a standalone model
4. **Generates** Turkish speech from the fine-tuned model

| Feature | Description |
| --- | --- |
| **LoRA fine-tuning** | Parameter-efficient adaptation (~5.6% trainable params) |
| **DDP distributed training** | Multi-node training via `TransformersTrainer` |
| **SNAC audio codec** | 24kHz waveform to discrete token conversion |
| **Audio-only loss** | Cross-entropy on audio tokens only; text tokens masked |

### Prerequisites

- OpenShift AI (RHOAI) 3.2+ with Kubeflow Trainer v2 enabled
- A workbench with GPU (for LoRA merge and inference)
- A shared RWX PVC named `shared` (60Gi recommended)
  - **Workbench mount**: `/opt/app-root/src/shared`

## Install the Kubeflow SDK

In [ ]:
!python3 -m pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2" \
    yamlmagic
%load_ext yamlmagic

## Training Configuration

Edit the following training parameters:

In [ ]:
%%yaml parameters

# Infrastructure
namespace: rhai-orpheus-tts      # set to your OpenShift AI project name
mlflow_experiment: orpheus-turkish-tts

# Model
base_model: unsloth/orpheus-3b-0.1-pretrained
hf_dataset: afkfatih/turkish-tts-combined-raw
max_train_samples: 2000                   # 0 = full dataset (~81K); 2000 for a quick run
max_seq_len: 4096

# LoRA
lora_r: 16
lora_alpha: 32
lora_dropout: 0                             # 0 is common for LoRA TTS; use 0.05 if overfitting

# Training
num_nodes: 2
gpus_per_node: 1
batch_size: 4                             # gradient checkpointing enabled — fits 4 on A100-80GB
grad_accum: 2                             # effective batch = batch_size × grad_accum × num_nodes × gpus_per_node
learning_rate: 1.0e-4                     # LoRA typically 1e-4–2e-4 (was 2e-5)
num_epochs: 3
eval_split: 0.05
warmup_ratio: 0.05

# Checkpointing & Logging
save_steps: 200                           # must be a multiple of eval_steps (auto-aligned if not)
logging_steps: 10
eval_steps: 100                           # loss eval cadence (cheap)
audio_log_steps: 500                      # wav + Whisper cadence (expensive) — keep sparse


## Training Loop

`TransformersTrainer` serializes `train_func` via `inspect.getsource()` — all logic must be **inside** the function body. The function below:

1. Defines `_preprocess()` — SNAC-encodes raw audio into token sequences (rank 0 only)
2. Waits for preprocessing on other ranks via a sentinel file
3. Runs the full training loop with LoRA, MLflow tracking, Whisper WER/CER evaluation, and audio artifact logging

> **Note:** `train_orpheus.py` contains the same logic for standalone CLI usage.


In [ ]:
def train_func(**parameters):  # noqa: C901
    """Self-contained training function — serialized via inspect.getsource()."""
    import hashlib
    import io
    import logging
    import math
    import os
    import sys
    import tempfile
    import time
    from pathlib import Path
    from types import SimpleNamespace

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[logging.StreamHandler(sys.stdout)],
    )
    log = logging.getLogger("train_orpheus")

    import warnings

    import urllib3

    warnings.filterwarnings("ignore", category=urllib3.exceptions.InsecureRequestWarning)

    import librosa
    import mlflow
    import numpy as np
    import soundfile as sf
    import torch
    from datasets import (
        Audio,
        Dataset,
        load_dataset,
        load_from_disk,
    )
    from peft import LoraConfig, TaskType, get_peft_model
    from scipy.signal import resample_poly
    from snac import SNAC
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForSeq2Seq,
        Trainer,
        TrainerCallback,
        TrainingArguments,
    )

    # ── Constants (Orpheus / SNAC token spec) ─────────────────────────────────
    LLAMA_VOCAB = 128_256
    CODE_OFFSET = LLAMA_VOCAB + 10
    N_CODEBOOK = 4_096
    N_PER_FRAME = 7
    SNAC_SR = 24_000

    TOK_SOH = LLAMA_VOCAB + 3
    TOK_EOH = LLAMA_VOCAB + 4
    TOK_SOA = LLAMA_VOCAB + 5
    TOK_EOA = LLAMA_VOCAB + 6
    TOK_SOS = LLAMA_VOCAB + 1
    TOK_EOT = LLAMA_VOCAB + 9

    # ── Parameters from %%yaml (via func_args) ─────────────────────────────────
    p = SimpleNamespace(
        audio_log_steps=500,
        eval_steps=100,
        preprocess_timeout_s=7200,
        min_preprocess_rows=100,
        whisper_model="small",
        gen_max_new_tokens=1500,
        gen_min_new_tokens=80,
        gen_baseline_max_new_cap=400,
        gen_temperature=0.3,
        gen_top_p=0.9,
        gen_repetition_penalty=1.15,
        gen_trim_top_db=28,
        gen_min_tokens_per_char=7,
        **parameters,
    )

    rank = int(os.environ.get("RANK", 0))
    pvc = "/mnt/kubeflow-checkpoints/orpheus-tts"
    hf_cache = f"{pvc}/hf-cache"
    data_dir = f"{pvc}/preprocessed"
    checkpoint_dir = f"{pvc}/checkpoints"

    EVAL_SENTENCES = [
        ("flight_announce", "sayın yolcularımız, uçuşumuz yaklaşık iki saat sürecektir."),
        ("welcome", "istanbul'a hoş geldiniz."),
        ("safety", "güvenlik nedeniyle elektronik cihazlarınızı kapalı tutunuz."),
        ("farewell", "teşekkür ederiz, iyi yolculuklar dileriz."),
    ]

    # ══════════════════════════════════════════════════════════════════════════
    # PHASE 1: PREPROCESSING (rank 0 only)
    # ══════════════════════════════════════════════════════════════════════════

    def _preprocess_fingerprint():
        """Invalidate cached preprocess when dataset / size / seq len change."""
        raw = f"{p.hf_dataset}|{p.max_train_samples}|{p.max_seq_len}|{p.base_model}"
        return hashlib.sha1(raw.encode()).hexdigest()[:12]

    def _preprocess():
        """SNAC-encode raw audio → token sequences. Writes versioned sentinel when done."""
        out_path = Path(data_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        fingerprint = _preprocess_fingerprint()
        sentinel = out_path / f".done-{fingerprint}"
        # Remove stale sentinels from older configs
        for stale in out_path.glob(".done*"):
            if stale.name != sentinel.name:
                stale.unlink(missing_ok=True)
        if sentinel.exists() and (out_path / "dataset_info.json").exists():
            log.info("Already preprocessed at %s (fp=%s) — skipping.", out_path, fingerprint)
            return

        os.environ["HF_HOME"] = hf_cache
        device = "cuda" if torch.cuda.is_available() else "cpu"
        log.info("Preprocessing on device: %s (fp=%s)", device, fingerprint)

        tokenizer = AutoTokenizer.from_pretrained(p.base_model, cache_dir=hf_cache)
        snac_model = (
            SNAC.from_pretrained("hubertsiuzdak/snac_24khz", cache_dir=hf_cache)
            .to(device)
            .eval()
        )

        log.info("Loading raw dataset: %s …", p.hf_dataset)
        raw = load_dataset(p.hf_dataset, split="train", cache_dir=hf_cache)
        raw = raw.cast_column("audio", Audio(decode=False))
        total = min(p.max_train_samples, len(raw)) if p.max_train_samples > 0 else len(raw)
        shard = raw.select(range(total))
        log.info("Processing %d samples", total)

        def _encode(wav_np, src_sr):
            if wav_np.ndim == 2:
                wav_np = wav_np.mean(axis=1)
            if src_sr != SNAC_SR:
                gcd = math.gcd(int(src_sr), SNAC_SR)
                wav_np = resample_poly(
                    wav_np, SNAC_SR // gcd, src_sr // gcd
                ).astype(np.float32)
            wav = torch.tensor(wav_np).unsqueeze(0).unsqueeze(0).to(device)
            with torch.inference_mode():
                codes = snac_model.encode(wav)
            return codes[0][0].tolist(), codes[1][0].tolist(), codes[2][0].tolist()

        def _interleave(l0, l1, l2):
            n = min(len(l0), len(l1) // 2, len(l2) // 4)
            t = []
            for f in range(n):
                t += [
                    CODE_OFFSET + l0[f],
                    CODE_OFFSET + 4096 + l1[2 * f],
                    CODE_OFFSET + 8192 + l2[4 * f],
                    CODE_OFFSET + 8192 + l2[4 * f + 1],
                    CODE_OFFSET + 4096 + l1[2 * f + 1],
                    CODE_OFFSET + 8192 + l2[4 * f + 2],
                    CODE_OFFSET + 8192 + l2[4 * f + 3],
                ]
            return t

        rows, skipped = [], 0
        for i, sample in enumerate(shard):
            if i % 500 == 0:
                log.info("  %d / %d  (skipped %d)", i, len(shard), skipped)
            try:
                text = sample["text"].strip()
                audio = sample["audio"]
                raw_bytes = audio.get("bytes") or open(audio["path"], "rb").read()
                wav, sr = sf.read(
                    io.BytesIO(raw_bytes), dtype="float32", always_2d=False
                )
                t_ids = tokenizer.encode(text, add_special_tokens=False)
                l0, l1, l2 = _encode(wav, sr)
                seq = (
                    [TOK_SOH]
                    + t_ids
                    + [TOK_EOT, TOK_EOH, TOK_SOA, TOK_SOS]
                    + _interleave(l0, l1, l2)
                    + [TOK_EOA]
                )
                if len(seq) > p.max_seq_len:
                    skipped += 1
                    continue
                sos_idx = seq.index(TOK_SOS)
                labels = [-100] * (sos_idx + 1) + seq[sos_idx + 1 :]
                rows.append(
                    {
                        "input_ids": seq,
                        "labels": labels,
                        "attention_mask": [1] * len(seq),
                    }
                )
            except Exception as e:
                log.warning("sample %d skipped: %s", i, e)
                skipped += 1

        min_rows = int(getattr(p, "min_preprocess_rows", 100))
        if len(rows) < min_rows:
            raise RuntimeError(
                f"Preprocessing kept only {len(rows)} sequences (need >= {min_rows}). "
                f"Skipped {skipped}. Check dataset schema (text/audio) or lower max_seq_len filters."
            )

        log.info("Preprocessing done: %d kept, %d skipped", len(rows), skipped)
        Dataset.from_list(rows).save_to_disk(str(out_path))
        sentinel.touch()
        log.info("Saved %d sequences → %s (fp=%s)", len(rows), out_path, fingerprint)

    # Rank 0: preprocess
    if rank == 0:
        _preprocess()

    # File-based sync — fail fast if preprocess never completes
    fingerprint = _preprocess_fingerprint()
    sentinel = Path(data_dir) / f".done-{fingerprint}"
    preprocess_timeout_s = int(getattr(p, "preprocess_timeout_s", 7200))  # 2h default
    waited = 0
    while not sentinel.exists():
        if waited >= preprocess_timeout_s:
            raise TimeoutError(
                f"Rank {rank}: preprocess sentinel {sentinel} not found after "
                f"{preprocess_timeout_s}s — rank 0 likely failed."
            )
        log.info(
            "Rank %d: waiting for preprocessing (fp=%s, %ds/%ds)...",
            rank, fingerprint, waited, preprocess_timeout_s,
        )
        time.sleep(30)
        waited += 30

    # ══════════════════════════════════════════════════════════════════════════
    # PHASE 2: TRAINING (all ranks)
    # ══════════════════════════════════════════════════════════════════════════

    os.environ["HF_HOME"] = hf_cache
    os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", p.mlflow_experiment)
    out_dir = Path(checkpoint_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Helpers ────────────────────────────────────────────────────────────────
    def build_prompt(tokenizer, text):
        ids = tokenizer.encode(text, add_special_tokens=False) + [TOK_EOT]
        return [TOK_SOH] + ids + [TOK_EOH, TOK_SOA, TOK_SOS]

    def _clean_wav(wav):
        wav = wav.astype(np.float32)
        try:
            trimmed, _ = librosa.effects.trim(
                wav, top_db=p.gen_trim_top_db, frame_length=2048, hop_length=512
            )
            if len(trimmed) / SNAC_SR >= 0.1:
                wav = trimmed
        except Exception:
            pass
        peak = np.abs(wav).max()
        if peak > 1e-6:
            wav = wav * (0.9 / peak)
        return wav.astype(np.float32)

    def snac_decode(snac_model, token_ids, device):
        audio_ids = [t for t in token_ids if t >= CODE_OFFSET]
        n = len(audio_ids) // N_PER_FRAME
        if n == 0:
            return None
        audio_ids = audio_ids[: n * N_PER_FRAME]
        l0, l1, l2 = [], [], []
        for f in range(n):
            g = audio_ids[N_PER_FRAME * f : N_PER_FRAME * (f + 1)]
            l0.append((g[0] - CODE_OFFSET) % N_CODEBOOK)
            l1.append((g[1] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[2] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[3] - CODE_OFFSET) % N_CODEBOOK)
            l1.append((g[4] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[5] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[6] - CODE_OFFSET) % N_CODEBOOK)

        def _t(x):
            return torch.tensor(x, dtype=torch.long).unsqueeze(0).to(device)

        wav = snac_model.decode([_t(l0), _t(l1), _t(l2)])
        return wav.squeeze().cpu().float().detach().numpy()

    def generate_audio(model, tokenizer, snac_model, text, device, step=0, *, baseline=False):
        t0 = time.perf_counter()
        prompt = build_prompt(tokenizer, text)
        inp = torch.tensor([prompt], dtype=torch.long, device=device)
        if baseline:
            min_new, max_new = p.gen_min_new_tokens, min(p.gen_max_new_tokens, p.gen_baseline_max_new_cap)
        else:
            min_new = min(
                p.gen_max_new_tokens - 50,
                max(p.gen_min_new_tokens, len(text) * p.gen_min_tokens_per_char),
            )
            max_new = p.gen_max_new_tokens
        with torch.inference_mode():
            out = model.generate(
                inp,
                max_new_tokens=max_new,
                min_new_tokens=min_new,
                do_sample=True,
                temperature=p.gen_temperature,
                top_p=p.gen_top_p,
                use_cache=True,
                repetition_penalty=p.gen_repetition_penalty,
                eos_token_id=TOK_EOA,
            )
        new_ids = out[0][len(prompt) :].cpu().tolist()
        if TOK_EOA in new_ids:
            new_ids = new_ids[: new_ids.index(TOK_EOA)]
        wav = snac_decode(snac_model, new_ids, device)
        elapsed = time.perf_counter() - t0
        if wav is not None:
            try:
                mlflow.log_metric(
                    "rtf_latest",
                    elapsed / max(len(wav) / SNAC_SR, 1e-6),
                    step=step,
                )
            except Exception:
                pass
        return wav, elapsed

    def compute_wer_cer(whisper_mdl, wav, reference):
        import jiwer

        hyp = whisper_mdl.transcribe(
            wav.astype(np.float32),
            language="tr",
            task="transcribe",
            initial_prompt="Türkçe konuşma.",
        )["text"].strip().lower()
        ref = reference.strip().lower()
        return jiwer.wer(ref, hyp), jiwer.cer(ref, hyp), hyp

    def log_audio_batch(
        model, tokenizer, snac_model, device, step, folder, whisper_mdl=None, *, baseline=False
    ):
        model.eval()
        metrics = {}
        tmp = Path(tempfile.mkdtemp())
        wers, cers, rtfs = [], [], []

        for label, text in EVAL_SENTENCES:
            wav, elapsed = generate_audio(
                model, tokenizer, snac_model, text, device, step=step, baseline=baseline
            )
            if wav is None:
                log.warning("No audio for '%s' at step %d", label, step)
                continue
            wav = _clean_wav(wav)
            duration = len(wav) / SNAC_SR
            if duration < 0.3:
                continue
            rtf = elapsed / max(duration, 1e-6)
            rtfs.append(rtf)
            metrics[f"rtf/{label}"] = rtf

            wav_path = tmp / f"{label}.wav"
            sf.write(str(wav_path), wav, samplerate=SNAC_SR)
            mlflow.log_artifact(str(wav_path), artifact_path=f"{folder}/{label}")

            if whisper_mdl is not None:
                try:
                    wer, cer, hyp = compute_wer_cer(whisper_mdl, wav, text)
                    wers.append(wer)
                    cers.append(cer)
                    metrics[f"wer/{label}"] = wer
                    metrics[f"cer/{label}"] = cer
                    log.info("  [%s] WER=%.3f  CER=%.3f  hyp: %s", label, wer, cer, hyp[:60])
                except Exception as e:
                    log.warning("WER/CER failed for %s: %s", label, e)

        if rtfs:
            metrics["rtf/mean"] = sum(rtfs) / len(rtfs)
        if wers:
            metrics["wer/mean"] = sum(wers) / len(wers)
        if cers:
            metrics["cer/mean"] = sum(cers) / len(cers)

        mlflow.log_metrics(metrics, step=step)
        model.train()

    # ── MLflow callback ───────────────────────────────────────────────────────
    class MLflowAudioCallback(TrainerCallback):
        def __init__(self):
            self._snac = None
            self._whisper = None
            self._device = None

        def _get_snac(self):
            if self._snac is None:
                self._snac = (
                    SNAC.from_pretrained("hubertsiuzdak/snac_24khz", cache_dir=hf_cache)
                    .eval()
                    .to(self._device)
                )
            return self._snac

        def _get_whisper(self):
            if self._whisper is None:
                import whisper

                self._whisper = whisper.load_model(p.whisper_model, device="cpu")
                log.info("Whisper-%s loaded on CPU for in-training WER/CER", p.whisper_model)
            return self._whisper

        def on_train_begin(self, args, state, control, model=None, **kwargs):
            self._device = next(model.parameters()).device
            if not state.is_world_process_zero:
                return
            world = int(os.environ.get("WORLD_SIZE", "1"))
            mlflow.set_tags(
                {
                    "base_model": p.base_model,
                    "dataset": p.hf_dataset,
                    "train_samples": str(p.max_train_samples or "full"),
                    "max_seq_len": str(p.max_seq_len),
                    "n_gpus": str(torch.cuda.device_count() * world),
                    "gpu": torch.cuda.get_device_name(0)
                    if torch.cuda.is_available()
                    else "cpu",
                    "effective_batch": str(p.batch_size * p.grad_accum * world),
                }
            )
            mlflow.log_params(
                {
                    "lora_r": p.lora_r,
                    "lora_alpha": p.lora_alpha,
                    "lora_dropout": p.lora_dropout,
                    "trainable_params_M": round(trainable / 1e6, 1),
                    "total_params_B": round(total / 1e9, 2),
                    "learning_rate": p.learning_rate,
                    "num_epochs": p.num_epochs,
                    "batch_size": p.batch_size,
                    "grad_accum_steps": p.grad_accum,
                    "dataset_samples": len(train_ds),
                }
            )
            log.info("Logging pretrained baseline audio + metrics …")
            try:
                log_audio_batch(
                    model,
                    tokenizer,
                    self._get_snac(),
                    self._device,
                    step=0,
                    folder="audio/step_000000/pretrained_baseline",
                    whisper_mdl=self._get_whisper(),
                    baseline=True,
                )
            except Exception as e:
                log.warning("Baseline audio failed: %s", e)

        def on_step_end(self, args, state, control, model=None, **kwargs):
            if not state.is_world_process_zero:
                return
            if state.global_step > 0 and state.global_step % p.audio_log_steps == 0:
                log.info("Logging audio at step %d …", state.global_step)
                try:
                    log_audio_batch(
                        model,
                        tokenizer,
                        self._get_snac(),
                        self._device,
                        step=state.global_step,
                        folder=f"audio/step_{state.global_step:06d}/finetuned",
                        whisper_mdl=self._get_whisper(),
                    )
                except Exception as e:
                    log.warning("Audio at step %d failed: %s", state.global_step, e)

        def on_train_end(self, args, state, control, model=None, **kwargs):
            if not state.is_world_process_zero:
                return
            try:
                log_audio_batch(
                    model,
                    tokenizer,
                    self._get_snac(),
                    self._device,
                    step=state.global_step,
                    folder="audio/final",
                    whisper_mdl=self._get_whisper(),
                )
            except Exception as e:
                log.warning("Final audio failed: %s", e)

    # ── Load tokenizer + model ────────────────────────────────────────────────
    log.info("Loading model: %s", p.base_model)
    tokenizer = AutoTokenizer.from_pretrained(p.base_model, cache_dir=hf_cache)
    tokenizer.pad_token = tokenizer.eos_token

    try:
        from flash_attn import flash_attn_func  # noqa: F401

        attn_impl = "flash_attention_2"
    except ImportError:
        attn_impl = "sdpa"
        log.warning("flash-attn not available — using SDPA")

    model = AutoModelForCausalLM.from_pretrained(
        p.base_model,
        cache_dir=hf_cache,
        torch_dtype=torch.bfloat16,
        attn_implementation=attn_impl,
    )
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=p.lora_r,
        lora_alpha=p.lora_alpha,
        lora_dropout=p.lora_dropout,
        bias="none",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, lora_cfg)
    model.enable_input_require_grads()
    trainable, total = model.get_nb_trainable_parameters()
    log.info(
        "LoRA: %.1fM trainable / %.2fB total (%.2f%%)",
        trainable / 1e6,
        total / 1e9,
        100 * trainable / total,
    )

    # ── Load preprocessed dataset ─────────────────────────────────────────────
    preprocessed_dir = Path(data_dir)
    if not sentinel.exists():
        raise RuntimeError(
            f"Preprocessed dataset sentinel missing: {sentinel}. "
            "Ensure preprocess() ran on rank 0 before calling train()."
        )
    log.info("Loading preprocessed dataset from %s", preprocessed_dir)
    ds = load_from_disk(str(preprocessed_dir))

    if p.max_train_samples:
        ds = ds.select(range(min(p.max_train_samples, len(ds))))

    def _audio_only_labels(example):
        ids = example["input_ids"]
        try:
            sos_idx = ids.index(TOK_SOS)
        except ValueError:
            example["labels"] = list(ids)
            return example
        example["labels"] = [-100] * (sos_idx + 1) + ids[sos_idx + 1 :]
        return example

    ds = ds.map(_audio_only_labels)
    ds = ds.filter(lambda x: len(x["input_ids"]) <= p.max_seq_len)
    ds = ds.map(lambda x: {"length": len(x["input_ids"])})

    split = ds.train_test_split(test_size=p.eval_split, seed=42)
    train_ds = split["train"]
    eval_ds = split["test"]
    log.info("Train: %d  Eval: %d", len(train_ds), len(eval_ds))

    # ── Trainer ───────────────────────────────────────────────────────────────
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding="longest",
        pad_to_multiple_of=8,
        label_pad_token_id=-100,
    )

    # Prefer fused AdamW; fall back on older torch builds
    import inspect as _inspect

    if "fused" in _inspect.signature(torch.optim.AdamW).parameters:
        optim_name = "adamw_torch_fused"
    else:
        optim_name = "adamw_torch"
        log.warning("adamw_torch_fused unavailable — using adamw_torch")

    # load_best_model_at_end requires save_steps to be a multiple of eval_steps
    eval_steps = int(p.eval_steps)
    save_steps = int(p.save_steps)
    if save_steps % eval_steps != 0:
        save_steps = ((save_steps + eval_steps - 1) // eval_steps) * eval_steps
        log.warning("Adjusted save_steps → %d to align with eval_steps=%d", save_steps, eval_steps)

    training_args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=p.num_epochs,
        per_device_train_batch_size=p.batch_size,
        per_device_eval_batch_size=p.batch_size,
        gradient_accumulation_steps=p.grad_accum,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=p.learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=p.warmup_ratio,
        weight_decay=0.01,
        optim=optim_name,
        bf16=True,
        tf32=True,
        logging_steps=p.logging_steps,
        eval_steps=eval_steps,
        eval_strategy="steps",
        save_steps=save_steps,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        group_by_length=True,
        length_column_name="length",
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        report_to="mlflow",
        run_name=f"orpheus-tr-{p.max_train_samples or 'full'}-b{p.batch_size}x{p.grad_accum}",
        ddp_find_unused_parameters=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        tokenizer=tokenizer,
        callbacks=[MLflowAudioCallback()],
    )

    log.info("Starting training …")
    trainer.train()

    if trainer.is_world_process_zero():
        final = out_dir / "final"
        trainer.save_model(str(final))
        tokenizer.save_pretrained(str(final))
        log.info("Model saved → %s", final)

        meta_files = [
            "config.json",
            "tokenizer_config.json",
            "tokenizer.json",
            "special_tokens_map.json",
        ]
        for fname in meta_files:
            meta_path = final / fname
            if meta_path.exists():
                mlflow.log_artifact(str(meta_path), artifact_path="model_metadata")

        mlflow.set_tags(
            {"final_checkpoint_path": str(final), "stage": "final"}
        )
        log.info("Final model tagged in MLflow: %s", final)


print("train_func defined — fully self-contained for inspect.getsource()")

## Training Client

Authenticate to the cluster API. In an OpenShift AI workbench, `NOTEBOOK_USER_TOKEN` is often set; otherwise the cell falls back to the pod service-account token.


In [ ]:
import os
from pathlib import Path

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

# Prefer workbench env vars; fall back to in-cluster API + SA token.
api_server = os.getenv("OPENSHIFT_API_URL", "https://kubernetes.default.svc")
token = os.getenv("NOTEBOOK_USER_TOKEN")
if not token:
    sa_token = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_token.exists():
        token = sa_token.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench with a service-account token."
    )

configuration = k8s.Configuration()
configuration.host = api_server
configuration.api_key = {"authorization": f"Bearer {token}"}
# Un-comment if your cluster API server uses a self-signed certificate
# configuration.verify_ssl = False

api_client = k8s.ApiClient(configuration)
client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=api_client.configuration,
    )
)


## Training Job

Create a `TransformersTrainer` with periodic checkpointing, JIT checkpointing, and progression tracking, then submit the TrainJob.


In [ ]:
import os

from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig

trainer = TransformersTrainer(
    func=train_func,
    func_args=parameters,
    num_nodes=parameters["num_nodes"],
    resources_per_node={
        "nvidia.com/gpu": parameters["gpus_per_node"],
        "cpu": "4",
        "memory": "32Gi",
    },
    env={
        # Prefer workbench / platform MLflow (set on the Notebook CR); fall back to in-cluster DNS
        "MLFLOW_TRACKING_URI": os.environ.get(
            "MLFLOW_TRACKING_URI",
            f"http://mlflow.{parameters['namespace']}.svc.cluster.local:5000",
        ),
        "MLFLOW_EXPERIMENT_NAME": parameters["mlflow_experiment"],
        "MLFLOW_TRACKING_INSECURE_TLS": "true",
    },
    packages_to_install=[
        "peft", "snac", "soundfile", "scipy", "librosa",
        "mlflow", "jiwer", "openai-whisper",
    ],
    output_dir="pvc://shared/orpheus-tts/checkpoints",
    periodic_checkpoint_config=PeriodicCheckpointConfig(
        save_strategy="steps",
        save_steps=parameters["save_steps"],
        save_total_limit=3,
    ),
    enable_jit_checkpoint=True,
    enable_progression_tracking=True,
)

runtime = client.backend.get_runtime("torch-distributed")
JOB_NAME = client.train(trainer=trainer, runtime=runtime)
print(f"Job submitted: {JOB_NAME}")


## Monitor the training job

`TransformersTrainer` injects a progress callback visible in **OpenShift AI Dashboard → Training Jobs**. Stream logs from the notebook:


In [ ]:
for logline in client.get_job_logs(JOB_NAME, follow=True):
    print(logline, end="")

In [ ]:
job = client.get_job(name=JOB_NAME)
print(f"Job: {job.name}")
print(f"Status: {job.status}")

## Merge LoRA adapter

After training completes, merge the LoRA adapter weights into the base model to produce a standalone model that can be used for inference without PEFT.


In [ ]:
import glob
import os

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

NOTEBOOK_SHARED = "/opt/app-root/src/shared"
hf_cache = f"{NOTEBOOK_SHARED}/orpheus-tts/hf-cache"
ckpt_base = f"{NOTEBOOK_SHARED}/orpheus-tts/checkpoints"
final_path = f"{NOTEBOOK_SHARED}/orpheus-tts/final"

final_ckpt = os.path.join(ckpt_base, "final")
if os.path.isdir(final_ckpt):
    best_ckpt = final_ckpt
else:
    checkpoints = sorted(glob.glob(os.path.join(ckpt_base, "checkpoint-*")))
    if not checkpoints:
        raise FileNotFoundError(
            f"No checkpoints found at {ckpt_base}. Ensure training completed successfully."
        )
    best_ckpt = checkpoints[-1]

print(f"Merging LoRA from: {best_ckpt}")

base_model_id = parameters["base_model"]
tokenizer = AutoTokenizer.from_pretrained(base_model_id, cache_dir=hf_cache)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id, cache_dir=hf_cache, torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(base, best_ckpt)
model = model.merge_and_unload()

os.makedirs(final_path, exist_ok=True)
model.save_pretrained(final_path, safe_serialization=True)
tokenizer.save_pretrained(final_path)

print(f"Merged model saved to {final_path}")
del model, base
torch.cuda.empty_cache()

## Results

> Screenshots below are from a reference run: **20K samples, 8 epochs, 2× A100-80GB, LoRA r=32/α=64**, tracked in MLflow on OpenShift AI.

### Training & eval loss

![Training Loss](images/training_loss.png)

### In-training WER/CER (Whisper ASR)

![WER CER Progress](images/wer_cer_progress.png)

### Full MLflow dashboard

![Dashboard](images/dashboard.png)

### MLflow Traces — inference pipeline

![MLflow Traces](images/ui_traces_pipeline.png)

### MLflow Artifacts — step-indexed audio

![MLflow Artifacts](images/ui_artifacts_audio.png)

### Post-training evaluation — baseline vs fine-tuned

| Metric | Baseline | Fine-tuned | Δ |
| --- | --- | --- | --- |
| **WER mean** | 1.576 | **0.723** | −0.854 |
| **CER mean** | 1.224 | **0.410** | −0.814 |
| **eval_loss** | 9.50 | **4.35** | −5.15 |

### Per-sentence WER & CER

![WER CER Bars](images/eval_wer_cer_bars.png)

### Per-sentence CER improvement (Δ)

![CER Delta](images/eval_cer_delta.png)

Audio samples available on the [HuggingFace model card](https://huggingface.co/AbDhumal/orpheus-3b-turkish-tts-v2#audio-samples).

## Generate Turkish speech

Load the merged model and generate audio for a few Turkish sentences to verify the fine-tuning worked.

The pipeline:
1. Tokenize Turkish text with the Llama-3 tokenizer
2. Generate SNAC audio tokens with the fine-tuned model
3. Decode tokens back to a 24kHz waveform via the SNAC codec

In [ ]:
!python3 -m pip install -q snac

import IPython.display as ipd
import torch
from snac import SNAC
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LLAMA_VOCAB = 128_256
CODE_OFFSET = LLAMA_VOCAB + 10
SNAC_SR = 24_000
TOK_SOH = LLAMA_VOCAB + 3
TOK_EOH = LLAMA_VOCAB + 4
TOK_SOA = LLAMA_VOCAB + 5
TOK_EOA = LLAMA_VOCAB + 6
TOK_SOS = LLAMA_VOCAB + 1
TOK_EOT = LLAMA_VOCAB + 9

N_CODEBOOK = 4096
N_PER_FRAME = 7

print("Loading fine-tuned model...")
ft_tokenizer = AutoTokenizer.from_pretrained(final_path)
ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_model = (
    AutoModelForCausalLM
    .from_pretrained(final_path, torch_dtype=torch.bfloat16)
    .eval()
    .to(device)
)

print("Loading SNAC decoder...")
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().to(device)


def build_prompt(text):
    ids = ft_tokenizer.encode(text, add_special_tokens=False) + [TOK_EOT]
    return [TOK_SOH] + ids + [TOK_EOH, TOK_SOA, TOK_SOS]


def snac_decode(token_ids):
    audio_ids = [t for t in token_ids if t >= CODE_OFFSET]
    n = len(audio_ids) // N_PER_FRAME
    if n == 0:
        return None
    audio_ids = audio_ids[: n * N_PER_FRAME]
    l0, l1, l2 = [], [], []
    for f in range(n):
        g = audio_ids[N_PER_FRAME * f : N_PER_FRAME * (f + 1)]
        l0.append((g[0] - CODE_OFFSET) % N_CODEBOOK)
        l1.append((g[1] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[2] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[3] - CODE_OFFSET) % N_CODEBOOK)
        l1.append((g[4] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[5] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[6] - CODE_OFFSET) % N_CODEBOOK)

    def _t(x):
        return torch.tensor(x, dtype=torch.long).unsqueeze(0).to(device)

    wav = snac_model.decode([_t(l0), _t(l1), _t(l2)])
    return wav.squeeze().cpu().float().detach().numpy()


def generate_speech(text, max_new_tokens=1500):
    prompt = build_prompt(text)
    inp = torch.tensor([prompt], dtype=torch.long, device=device)
    with torch.inference_mode():
        out = ft_model.generate(
            inp,
            max_new_tokens=max_new_tokens,
            min_new_tokens=max(80, len(text) * 7),
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            eos_token_id=TOK_EOA,
        )
    new_ids = out[0][len(prompt) :].cpu().tolist()
    if TOK_EOA in new_ids:
        new_ids = new_ids[: new_ids.index(TOK_EOA)]
    return snac_decode(new_ids)


# Test sentences
test_sentences = [
    ("welcome", "istanbul'a hos geldiniz."),
    ("flight", "sayin yolcularimiz, ucusumuz yaklasik iki saat surecektir."),
    ("farewell", "tesekkur ederiz, iyi yolculuklar dileriz."),
]

for label, text in test_sentences:
    print(f"\n[{label}] {text}")
    wav = generate_speech(text)
    if wav is not None:
        duration = len(wav) / SNAC_SR
        print(f"  Duration: {duration:.2f}s")
        ipd.display(ipd.Audio(wav, rate=SNAC_SR))
    else:
        print("  Failed to generate audio")

## Cleanup

Delete the training job to free cluster resources.

The model, dataset, and checkpoints remain on the PVC for future use.

In [ ]:
client.delete_job(name=JOB_NAME)
print(f"Job {JOB_NAME} deleted")

## Summary

You have fine-tuned Orpheus-3B for Turkish TTS using LoRA on Red Hat OpenShift AI.

For production-quality results, increase `max_train_samples` to 20,000+, `num_epochs` to 8, and use `lora_r: 32` / `lora_alpha: 64` in the YAML config. MLflow tracking is enabled by default — browse the experiment named in `mlflow_experiment` (platform MLflow UI, or your in-cluster tracker).

See the other examples in `examples/trainer/` for FSDP, DeepSpeed, Kueue, and S3 checkpoint storage.
